# GPTQ量化比较

In [1]:
import gc
import json
import math
import os
import time
from pathlib import Path

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from gptqmodel import BACKEND, GPTQModel

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


W1102 22:37:41.875000 53875 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1102 22:37:41.998000 53875 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.7.0
Transformers : 5.20.2
Torch        : 2.14.1+cu130
Triton       : 3.8.0


In [2]:
DEVICE = "cuda:0"
DTYPE = torch.float16

# 配置四个模型
MODEL_SPECS = [
    {
        "name": "OPT-2.7B FP16",
        "path": "facebook/opt-2.7b",
        "tokenizer_id": "facebook/opt-2.7b",
        "quantized": False,
        "disk_path": None,
    },
    {
        "name": "OPT-2.7B GPTQ-4bit",
        "path": (
            "/home/sigan/work/LLM-quickstart/"
            "quantization/models/opt-2.7b-gptq"
        ),
        "tokenizer_id": "facebook/opt-2.7b",
        "quantized": True,
        "disk_path": (
            "/home/sigan/work/LLM-quickstart/"
            "quantization/models/opt-2.7b-gptq"
        ),
    },
    {
        "name": "OPT-6.7B FP16",
        "path": (
            "/sigan/ai-ml/models/dense/opt-6.7b"
        ),
        "tokenizer_id": (
            "/sigan/ai-ml/models/dense/opt-6.7b"
        ),
        "quantized": False,
        "disk_path": (
            "/sigan/ai-ml/models/dense/opt-6.7b"
        ),
    },
    {
        "name": "OPT-6.7B GPTQ-4bit",
        "path": (
            "/sigan/ai-ml/models/quantized/"
            "opt-6.7b-gptqmodel-4bit-local"
        ),
        "tokenizer_id": (
            "/sigan/ai-ml/models/dense/opt-6.7b"
        ),
        "quantized": True,
        "disk_path": (
            "/sigan/ai-ml/models/quantized/"
            "opt-6.7b-gptqmodel-4bit-local"
        ),
    },
]

## 固定测试提示词

In [3]:
PROMPTS = [
    "Merry Christmas! I'm glad to",
    "The capital of France is",
    "Machine learning is",
    "In the future, artificial intelligence will",
    "A good way to learn programming is",
]

MAX_NEW_TOKENS = 64
PPL_WINDOW_SIZE = 512
PPL_WINDOWS = 16

## 准备固定 PPL 数据

In [4]:
wiki_test = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="test",
)

fixed_text = "\n\n".join(
    row.strip()
    for row in wiki_test["text"]
    if row.strip()
)

INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md?%2Fdatasets%2FSalesforce%2Fwikitext%2Fresolve%2Fmain%2FREADME.md=&etag=%222a4fec2bc8df76c9d4da1c8e8865b625eb221c76%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/wikitext/wikit

## 辅助函数

In [5]:
def directory_size_gb(path):
    if path is None:
        return None

    target = Path(path)

    if not target.exists():
        return None

    return sum(
        file.stat().st_size
        for file in target.rglob("*")
        if file.is_file()
    ) / 1024**3

In [6]:
def load_model(spec):
    if spec["quantized"]:
        wrapper = GPTQModel.load(
            str(spec["path"]),
            device=DEVICE,
            dtype=DTYPE,
            backend=BACKEND.GPTQ_TORCH,
        )

        model = wrapper.model
        return model, wrapper

    model = AutoModelForCausalLM.from_pretrained(
        str(spec["path"]),
        torch_dtype=DTYPE,
        device_map={"": 0},
    )

    return model, None

In [7]:
def calculate_perplexity(
    model,
    tokenizer,
):
    encoded = tokenizer(
        fixed_text,
        return_tensors="pt",
        truncation=True,
        max_length=PPL_WINDOW_SIZE * PPL_WINDOWS,
    )

    input_ids = encoded["input_ids"][0]
    total_nll = 0.0
    total_tokens = 0

    model.eval()

    with torch.no_grad():
        for start in range(
            0,
            len(input_ids),
            PPL_WINDOW_SIZE,
        ):
            window = input_ids[
                start:start + PPL_WINDOW_SIZE
            ]

            if window.numel() < 2:
                continue

            batch = window.unsqueeze(0).to(
                DEVICE
            )

            outputs = model(
                input_ids=batch,
                labels=batch,
            )

            token_count = window.numel() - 1

            total_nll += (
                outputs.loss.item()
                * token_count
            )
            total_tokens += token_count

    if total_tokens == 0:
        raise RuntimeError("PPL 测试 token 数量为 0")

    return math.exp(
        total_nll / total_tokens
    )

In [8]:
def run_generation_benchmark(
    model,
    tokenizer,
):
    model.eval()

    # 预热一次，避免首次 kernel 初始化影响计时
    warmup_inputs = tokenizer(
        "Hello",
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        model.generate(
            **warmup_inputs,
            max_new_tokens=8,
            do_sample=False,
        )

    torch.cuda.synchronize()

    outputs = {}
    generated_tokens = 0
    total_generation_time = 0.0

    for prompt in PROMPTS:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(DEVICE)

        input_length = inputs[
            "input_ids"
        ].shape[1]

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=(
                    tokenizer.eos_token_id
                ),
            )

        torch.cuda.synchronize()
        elapsed = (
            time.perf_counter()
            - start_time
        )

        new_tokens = (
            output_ids.shape[1]
            - input_length
        )

        generated_tokens += new_tokens
        total_generation_time += elapsed

        outputs[prompt] = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True,
        )

    tokens_per_second = (
        generated_tokens
        / total_generation_time
        if total_generation_time > 0
        else None
    )

    return (
        outputs,
        total_generation_time,
        tokens_per_second,
    )

## 依次测试四个模型

In [9]:
results = []
fixed_outputs = {}

for spec in MODEL_SPECS:
    name = spec["name"]

    print("=" * 80)
    print("开始测试：", name)

    loaded_model = None
    wrapper = None

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            spec["tokenizer_id"],
            local_files_only=Path(
                spec["tokenizer_id"]
            ).exists(),
        )

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        load_start = time.perf_counter()

        loaded_model, wrapper = load_model(
            spec
        )

        torch.cuda.synchronize()

        load_seconds = (
            time.perf_counter()
            - load_start
        )

        load_peak_allocated = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        load_peak_reserved = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        loaded_model.eval()

        model_footprint = (
            loaded_model.get_memory_footprint()
            / 1024**3
        )

        parameter_count = (
            loaded_model.num_parameters()
        )

        print("计算 PPL...")
        ppl = calculate_perplexity(
            loaded_model,
            tokenizer,
        )

        torch.cuda.reset_peak_memory_stats()

        print("执行固定提示词生成...")
        (
            generated_outputs,
            generation_seconds,
            tokens_per_second,
        ) = run_generation_benchmark(
            loaded_model,
            tokenizer,
        )

        generation_peak_allocated = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        generation_peak_reserved = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        fixed_outputs[name] = generated_outputs

        results.append(
            {
                "model": name,
                "status": "ok",
                "parameters_billion": (
                    parameter_count / 1e9
                ),
                "ppl": ppl,
                "model_footprint_gb": (
                    model_footprint
                ),
                "disk_size_gb": (
                    directory_size_gb(
                        spec["disk_path"]
                    )
                ),
                "load_seconds": load_seconds,
                "load_peak_allocated_gb": (
                    load_peak_allocated
                ),
                "load_peak_reserved_gb": (
                    load_peak_reserved
                ),
                "generation_seconds": (
                    generation_seconds
                ),
                "tokens_per_second": (
                    tokens_per_second
                ),
                "generation_peak_allocated_gb": (
                    generation_peak_allocated
                ),
                "generation_peak_reserved_gb": (
                    generation_peak_reserved
                ),
            }
        )

    except Exception as error:
        results.append(
            {
                "model": name,
                "status": "failed",
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
            }
        )

        print("测试失败：", name)
        print(type(error).__name__, error)

    finally:
        del loaded_model
        del wrapper
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

开始测试： OPT-2.7B FP16


INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-2.7b/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/commits/main "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/discussion

计算 PPL...
执行固定提示词生成...
开始测试： OPT-2.7B GPTQ-4bit


INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


from_quantized: adapter: None
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: batch_size.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: block_name_to_quantize.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: cache_block_outputs.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: dataset.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: exllama_config.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: max_input_length.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_seqlen.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: module_name_preceding_first_block.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: modules_in_block_to_quantize.
INFO  QuantizeConfig: Ignoring unkn

INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='').


... (7 outputs left out)

INFO  The layer model.decoder.embed_tokens is not quantized.                    
INFO  The layer model.decoder.embed_positions is not quantized.                 
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Kernel: candidates -> `[TorchLinear]`                                     
INFO  Kernel: per-module selections -> `[TorchLinear=192]`                      
INFO  Kernel: selected -> `TorchLinear`.                                        
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Format: Converting `format` from `gptq` to internal `gptq_v2`.            
INFO  Kernel: selected: `TorchLinear`                                           
INFO  gc.collect() reclaimed 96 objects in 0.138s                               


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[TorchLinear]`                                         
计算 PPL...
执行固定提示词生成...


## 查看汇总结果

In [10]:
print(
    json.dumps(
        results,
        ensure_ascii=False,
        indent=2,
    )
)

[
  {
    "model": "OPT-2.7B FP16",
    "status": "ok",
    "parameters_billion": 2.6515968,
    "ppl": 17.139075461831908,
    "model_footprint_gb": 4.938983917236328,
    "disk_size_gb": null,
    "load_seconds": 42.659782257003826,
    "load_peak_allocated_gb": 5.298559188842773,
    "load_peak_reserved_gb": 5.31640625,
    "generation_seconds": 2.9646433680172777,
    "tokens_per_second": 94.78374465928758,
    "generation_peak_allocated_gb": 4.969149589538574,
    "generation_peak_reserved_gb": 5.595703125
  },
  {
    "model": "OPT-2.7B GPTQ-4bit",
    "status": "ok",
    "parameters_billion": 0.13427712,
    "ppl": 17.550603619963173,
    "model_footprint_gb": 1.471893310546875,
    "disk_size_gb": 1.4720020294189453,
    "load_seconds": 2.191856628007372,
    "load_peak_allocated_gb": 1.4799079895019531,
    "load_peak_reserved_gb": 1.49609375,
    "generation_seconds": 8.131080402003136,
    "tokens_per_second": 33.57487400232139,
    "generation_peak_allocated_gb": 1.55064296

In [11]:
for output in results:
    print("=" * 80)

    for key, value in output.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

model: OPT-2.7B FP16
status: ok
parameters_billion: 2.6516
ppl: 17.1391
model_footprint_gb: 4.9390
disk_size_gb: None
load_seconds: 42.6598
load_peak_allocated_gb: 5.2986
load_peak_reserved_gb: 5.3164
generation_seconds: 2.9646
tokens_per_second: 94.7837
generation_peak_allocated_gb: 4.9691
generation_peak_reserved_gb: 5.5957
model: OPT-2.7B GPTQ-4bit
status: ok
parameters_billion: 0.1343
ppl: 17.5506
model_footprint_gb: 1.4719
disk_size_gb: 1.4720
load_seconds: 2.1919
load_peak_allocated_gb: 1.4799
load_peak_reserved_gb: 1.4961
generation_seconds: 8.1311
tokens_per_second: 33.5749
generation_peak_allocated_gb: 1.5506
generation_peak_reserved_gb: 2.1289
model: OPT-6.7B FP16
status: ok
parameters_billion: 6.6585
ppl: 14.6679
model_footprint_gb: 12.4024
disk_size_gb: 37.2092
load_seconds: 120.8644
load_peak_allocated_gb: 12.4103
load_peak_reserved_gb: 12.4258
generation_seconds: 6.2462
tokens_per_second: 43.0660
generation_peak_allocated_gb: 12.4458
generation_peak_reserved_gb: 13.2695
m

In [12]:
# 查看固定样例
for model_name, outputs in fixed_outputs.items():
    print("=" * 80)
    print(model_name)

    for prompt, generated in outputs.items():
        print("-" * 60)
        print("Prompt：", prompt)
        print("Output：", generated)

OPT-2.7B FP16
------------------------------------------------------------
Prompt： Merry Christmas! I'm glad to
Output： Merry Christmas! I'm glad to see you're still around.
I'm still around, just not posting as much. I'm still here though.
------------------------------------------------------------
Prompt： The capital of France is
Output： The capital of France is Paris.
I'm not sure if you're being sarcastic or not, but I'm going to assume you're not.                                       
------------------------------------------------------------
Prompt： Machine learning is
Output： Machine learning is a powerful tool for analyzing data. It can be used to predict the future, to identify patterns, and to make predictions about the future. Machine learning is a powerful tool for analyzing data. It can be used to predict the future, to identify patterns, and to make predictions about the future.

Machine learning is
------------------------------------------------------------
Prompt： 

In [13]:
# 自动比较同尺寸模型
def find_result(name):
    for row in results:
        if row["model"] == name:
            return row
    return None


pairs = [
    (
        "OPT-2.7B FP16",
        "OPT-2.7B GPTQ-4bit",
    ),
    (
        "OPT-6.7B FP16",
        "OPT-6.7B GPTQ-4bit",
    ),
]

for fp16_name, quant_name in pairs:
    fp16 = find_result(fp16_name)
    quant = find_result(quant_name)

    print("=" * 80)
    print(fp16_name, "→", quant_name)

    if not fp16 or not quant:
        print("缺少对比结果")
        continue

    if (
        fp16["status"] != "ok"
        or quant["status"] != "ok"
    ):
        print("至少一个模型测试失败")
        continue

    print(
        "PPL：",
        fp16["ppl"],
        "→",
        quant["ppl"],
        "变化：",
        (
            quant["ppl"]
            - fp16["ppl"]
        )
        / fp16["ppl"]
        * 100,
        "%",
    )

    print(
        "峰值显存 allocated：",
        fp16["generation_peak_allocated_gb"],
        "→",
        quant["generation_peak_allocated_gb"],
    )

    print(
        "生成速度 tokens/s：",
        fp16["tokens_per_second"],
        "→",
        quant["tokens_per_second"],
    )

    if (
        fp16["disk_size_gb"]
        and quant["disk_size_gb"]
    ):
        print(
            "磁盘大小 GB：",
            fp16["disk_size_gb"],
            "→",
            quant["disk_size_gb"],
            "压缩比：",
            (
                fp16["disk_size_gb"]
                / quant["disk_size_gb"]
            ),
        )

OPT-2.7B FP16 → OPT-2.7B GPTQ-4bit
PPL： 17.139075461831908 → 17.550603619963173 变化： 2.401110602772729 %
峰值显存 allocated： 4.969149589538574 → 1.550642967224121
生成速度 tokens/s： 94.78374465928758 → 33.57487400232139
OPT-6.7B FP16 → OPT-6.7B GPTQ-4bit
PPL： 14.66787949127304 → 14.909245140935205 变化： 1.64553881019933 %
峰值显存 allocated： 12.445760726928711 → 3.691915512084961
生成速度 tokens/s： 43.06595505878156 → 23.232469581885997
磁盘大小 GB： 37.2092196829617 → 3.5274024195969105 压缩比： 10.548617723977674
